# Join additive ALT dosage to clinical data

**Question:** How are genotype dosage and interaction terms joined to each clinical cohort?

**Inputs:** Q3 CVD/CRP model inputs and an existing validated genotype-wide CSV.

**Outputs:** 04_cvd_additive_genotype_model_dataset.csv and 05_crp_additive_genotype_model_dataset.csv.

Hail extraction is a separate preceding stage. This notebook now stops if genotype inputs are missing rather than starting an alternative extraction route. ALT dosage is a coding label, not an established functional or risk-allele direction.


## Configuration


In [ ]:
import os
import sys
import math
import traceback
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

TARGET_SNPS = [
    {"rsid": "rs731236", "label": "TaqI", "contig": "12", "position": 47844974, "ref": "A", "alt": "G", "expected_vid": "12-47844974-A-G"},
    {"rsid": "rs7975232", "label": "ApaI", "contig": "12", "position": 47845054, "ref": "C", "alt": "A", "expected_vid": "12-47845054-C-A"},
    {"rsid": "rs1544410", "label": "BsmI", "contig": "12", "position": 47846052, "ref": "C", "alt": "T", "expected_vid": "12-47846052-C-T"},
    {"rsid": "rs2228570", "label": "FokI", "contig": "12", "position": 47879112, "ref": "A", "alt": "G", "expected_vid": "12-47879112-A-G"},
]
TIER1_RSIDS = [x["rsid"] for x in TARGET_SNPS]
RUN_ID = os.environ.get("Q3_VDR_RUN_ID", datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))

def infer_default_data_dir():
    cwd = Path.cwd()
    candidates = [
        Path("/home/dataproc/workspace/VDR"),
        Path("/home/jupyter/workspace/VDR"),
        Path("/home/jupyter/workspace/VDR/data"),
        Path("/home/jupyter/workspace/Q3_VDR"),
        cwd,
        cwd / "data",
        cwd.parent / "data",
    ]
    for candidate in candidates:
        try:
            if candidate.exists() and os.access(candidate, os.W_OK):
                return candidate
        except Exception:
            pass
    return cwd

DATA_DIR = Path(os.environ.get("Q3_VDR_DATA_DIR", str(infer_default_data_dir()))).expanduser()

def resolve_existing_dir(env_name, default_dir_name):
    manual = os.environ.get(env_name, "").strip()
    if manual:
        return Path(manual).expanduser()
    roots = []
    for root in [DATA_DIR, Path("/home/jupyter/workspace/VDR"), Path("/home/dataproc/workspace/VDR"), Path("/home/jupyter/workspace"), Path("/home/dataproc/workspace"), Path.cwd(), Path.cwd().parent]:
        if root not in roots:
            roots.append(root)
    dir_names = [default_dir_name]
    if default_dir_name == "Q3_VDR_v9_model_prep_outputs":
        dir_names.append("prepresult")
    marker = "02_cvd_model_dataset_v9.csv" if default_dir_name == "Q3_VDR_v9_model_prep_outputs" else "14_hail_extraction_status.csv"
    for root in roots:
        for dir_name in dir_names:
            candidate = root / dir_name
            try:
                if candidate.exists() and (candidate / marker).exists():
                    return candidate
            except Exception:
                pass
    for root in roots:
        try:
            if root.exists():
                hits = sorted(root.rglob(marker))
                if hits:
                    return hits[0].parent
        except Exception:
            pass
    return DATA_DIR / default_dir_name

MODEL_PREP_DIR = resolve_existing_dir("Q3_VDR_MODEL_OUTPUT_DIR", "Q3_VDR_v9_model_prep_outputs")
HAIL_OUTPUT_DIR = resolve_existing_dir("Q3_VDR_HAIL_OUTPUT_DIR", "Q3_VDR_tier1_hail_genotype_outputs_2026_08_07")
OUTPUT_DIR = Path(os.environ.get("Q3_VDR_ADDITIVE_OUTPUT_DIR", str(DATA_DIR / "Q3_VDR_tier1_additive_genotype_outputs_2026_08_08"))).expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OVERWRITE_OUTPUTS = os.environ.get("Q3_VDR_OVERWRITE_OUTPUTS", "true").strip().lower() in {"1", "true", "yes", "y"}
USE_EXISTING_GENOTYPE_WIDE = True
RUN_TARGETED_HAIL_EXTRACTION = False
STOP_IF_NO_GENOTYPE = True
MIN_MODEL_N = int(os.environ.get("Q3_VDR_15_MIN_MODEL_N", "100"))
MIN_CVD_EVENTS = int(os.environ.get("Q3_VDR_15_MIN_CVD_EVENTS", "50"))

GOOGLE_PROJECT = os.environ.get("GOOGLE_PROJECT", os.environ.get("GOOGLE_CLOUD_PROJECT", "")).strip()
WORKSPACE_BUCKET = os.environ.get("WORKSPACE_BUCKET", "").strip()
HAIL_TMP_DIR = os.environ.get("Q3_VDR_HAIL_TMP_DIR", (WORKSPACE_BUCKET.rstrip("/") + "/hail_tmp/q3_vdr_15") if WORKSPACE_BUCKET.startswith("gs://") else "").strip()
HAIL_PATH_CANDIDATES = []
for source, path in [
    ("Q3_VDR_HAIL_MT_PATH", os.environ.get("Q3_VDR_HAIL_MT_PATH", "").strip()),
    ("WGS_ACAF_THRESHOLD_SPLIT_HAIL_PATH", os.environ.get("WGS_ACAF_THRESHOLD_SPLIT_HAIL_PATH", "").strip()),
    ("WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH", os.environ.get("WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH", "").strip()),
    ("AOU_V9_WGS_ACAF_THRESHOLD_SPLIT_HAIL_PATH_FALLBACK", "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/splitMT/hail.mt"),
    ("AOU_V9_WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH_FALLBACK", "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/multiMT/hail.mt"),
]:
    if path and path not in {p for _, p in HAIL_PATH_CANDIDATES}:
        HAIL_PATH_CANDIDATES.append((source, path))

written_outputs = []
status_rows = []

def save(df, filename):
    path = OUTPUT_DIR / filename
    if path.exists() and not OVERWRITE_OUTPUTS:
        raise FileExistsError(f"{path} exists and Q3_VDR_OVERWRITE_OUTPUTS is false")
    df.to_csv(path, index=False)
    written_outputs.append(path)
    print(f"Saved {filename}: {len(df):,} rows")
    return path

def save_text(text, filename):
    path = OUTPUT_DIR / filename
    if path.exists() and not OVERWRITE_OUTPUTS:
        raise FileExistsError(f"{path} exists and Q3_VDR_OVERWRITE_OUTPUTS is false")
    path.write_text(text, encoding="utf-8")
    written_outputs.append(path)
    print(f"Saved {filename}: {len(text):,} characters")
    return path

def add_status(step, status, detail=""):
    row = {"run_id": RUN_ID, "step": step, "status": status, "detail": str(detail)[:4000]}
    status_rows.append(row)
    return row

def pct(n, denom):
    return round(100 * n / denom, 3) if denom else np.nan

save(pd.DataFrame([{
    "run_id": RUN_ID,
    "data_dir": str(DATA_DIR),
    "model_prep_dir": str(MODEL_PREP_DIR),
    "hail_output_dir": str(HAIL_OUTPUT_DIR),
    "output_dir": str(OUTPUT_DIR),
    "use_existing_genotype_wide": USE_EXISTING_GENOTYPE_WIDE,
    "run_targeted_hail_extraction": RUN_TARGETED_HAIL_EXTRACTION,
    "stop_if_no_genotype": STOP_IF_NO_GENOTYPE,
    "hail_path_candidates": ";".join([f"{s}={p}" for s, p in HAIL_PATH_CANDIDATES]),
}]), "00_additive_run_configuration.csv")
print("MODEL_PREP_DIR:", MODEL_PREP_DIR)
print("HAIL_OUTPUT_DIR:", HAIL_OUTPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


## Load clinical datasets


In [ ]:
def read_csv_optional(path, label):
    row = {"run_id": RUN_ID, "label": label, "path": str(path), "exists": path.exists(), "rows": 0, "cols": 0, "status": "missing"}
    if not path.exists():
        return pd.DataFrame(), row
    try:
        df = pd.read_csv(path)
        row.update({"rows": len(df), "cols": len(df.columns), "status": "loaded"})
        return df, row
    except Exception:
        row.update({"status": "read_error", "detail": traceback.format_exc()[:2000]})
        return pd.DataFrame(), row

cvd, cvd_input = read_csv_optional(MODEL_PREP_DIR / "02_cvd_model_dataset_v9.csv", "CVD model-prep dataset")
crp, crp_input = read_csv_optional(MODEL_PREP_DIR / "03_crp_model_dataset_v9.csv", "CRP model-prep dataset")
save(pd.DataFrame([cvd_input, crp_input]), "01_additive_input_dataset_status.csv")

for name, df in [("cvd", cvd), ("crp", crp)]:
    if not df.empty and "person_id" in df.columns:
        df["person_id"] = pd.to_numeric(df["person_id"], errors="coerce").astype("Int64")
    elif not df.empty:
        add_status(f"{name}_person_id", "missing", "person_id column not found")

analytic_ids = set()
for df in [cvd, crp]:
    if not df.empty and "person_id" in df.columns:
        analytic_ids.update([str(int(x)) for x in df["person_id"].dropna().unique()])

if not analytic_ids:
    add_status("load_model_prep_inputs", "blocked", "No analytic person_id found. Check Q3_VDR_MODEL_OUTPUT_DIR and model-prep outputs.")
else:
    add_status("load_model_prep_inputs", "pass", f"cvd_rows={len(cvd)}; crp_rows={len(crp)}; analytic_ids={len(analytic_ids)}")

save(pd.DataFrame([{
    "run_id": RUN_ID,
    "cvd_rows": len(cvd),
    "crp_rows": len(crp),
    "analytic_person_id_union_n": len(analytic_ids),
    "cvd_has_person_id": (not cvd.empty and "person_id" in cvd.columns),
    "crp_has_person_id": (not crp.empty and "person_id" in crp.columns),
}]), "02_additive_analytic_person_id_summary.csv")


## Load validated genotype dosage


In [ ]:
def normalize_genotype_wide(df):
    if df.empty:
        return df
    d = df.copy()
    if "person_id" not in d.columns:
        if "sample_id" in d.columns:
            d["person_id"] = pd.to_numeric(d["sample_id"], errors="coerce").astype("Int64")
        else:
            d["person_id"] = pd.Series([pd.NA] * len(d), dtype="Int64")
    else:
        d["person_id"] = pd.to_numeric(d["person_id"], errors="coerce").astype("Int64")
    for rsid in TIER1_RSIDS:
        if rsid in d.columns and f"{rsid}_alt_dosage" not in d.columns:
            d[f"{rsid}_alt_dosage"] = pd.to_numeric(d[rsid], errors="coerce")
        elif f"{rsid}_alt_dosage" in d.columns:
            d[f"{rsid}_alt_dosage"] = pd.to_numeric(d[f"{rsid}_alt_dosage"], errors="coerce")
    keep = ["person_id"] + (["sample_id"] if "sample_id" in d.columns else []) + [f"{rsid}_alt_dosage" for rsid in TIER1_RSIDS if f"{rsid}_alt_dosage" in d.columns]
    return d[keep].drop_duplicates(subset=["person_id"]).copy()

def find_existing_genotype_wide():
    manual = os.environ.get("Q3_VDR_15_GENOTYPE_WIDE_PATH", "").strip()
    candidates = []
    if manual:
        candidates.append(Path(manual).expanduser())
    candidates.extend([HAIL_OUTPUT_DIR / "14_tier1_vdr_genotype_wide.csv", OUTPUT_DIR / "15_tier1_vdr_genotype_wide_analytic.csv"])
    for p in candidates:
        if p.exists():
            return p
    return None

genotype_source = "none"
genotype_wide = pd.DataFrame()
genotype_long = pd.DataFrame()
existing_wide_path = find_existing_genotype_wide() if USE_EXISTING_GENOTYPE_WIDE else None

if existing_wide_path is not None:
    try:
        genotype_wide = normalize_genotype_wide(pd.read_csv(existing_wide_path))
        genotype_source = f"existing_wide:{existing_wide_path}"
        add_status("load_existing_genotype_wide", "pass", f"rows={len(genotype_wide)}; path={existing_wide_path}")
    except Exception:
        add_status("load_existing_genotype_wide", "fail", traceback.format_exc()[:2000])


if genotype_wide.empty:
    add_status("genotype_available_for_modeling", "blocked", "No person-level additive genotype wide data available.")
else:
    genotype_wide = normalize_genotype_wide(genotype_wide)
    add_status("genotype_available_for_modeling", "pass", f"rows={len(genotype_wide)}; source={genotype_source}")

save(pd.DataFrame(status_rows), "15_additive_extraction_status.csv")
save(pd.DataFrame([{
    "run_id": RUN_ID,
    "genotype_source": genotype_source,
    "genotype_wide_rows": len(genotype_wide),
    "has_all_four_dosage_columns": all(f"{rsid}_alt_dosage" in genotype_wide.columns for rsid in TIER1_RSIDS),
}]), "03_additive_genotype_source_status.csv")
if genotype_wide.empty and STOP_IF_NO_GENOTYPE:
    raise RuntimeError("No person-level genotype is available for additive modeling.")


## Join dosage and summarize analytic completeness


In [ ]:
def add_dosage_terms(df):
    if df.empty or genotype_wide.empty or "person_id" not in df.columns:
        return df.copy()
    gcols = ["person_id"] + [f"{rsid}_alt_dosage" for rsid in TIER1_RSIDS if f"{rsid}_alt_dosage" in genotype_wide.columns]
    d = df.merge(genotype_wide[gcols], on="person_id", how="left", validate="many_to_one")
    for col in ["vitd_lt20", "log_vitd", "vitd_per10", "age_at_index", "female", "winter", "bmi_full", "follow_up_years", "event_mi_or_stroke", "log_crp"]:
        if col in d.columns:
            d[col] = pd.to_numeric(d[col], errors="coerce")
    for rsid in TIER1_RSIDS:
        dosage = f"{rsid}_alt_dosage"
        if dosage in d.columns:
            d[dosage] = pd.to_numeric(d[dosage], errors="coerce")
            d[f"{rsid}_alt_carrier_from_dosage"] = np.where(d[dosage].notna(), (d[dosage] > 0).astype(int), np.nan)
            d[f"{rsid}_x_vitd_lt20_additive"] = d[dosage] * d["vitd_lt20"] if "vitd_lt20" in d.columns else np.nan
            d[f"{rsid}_x_log_vitd_additive"] = d[dosage] * d["log_vitd"] if "log_vitd" in d.columns else np.nan
    return d

cvd_add = add_dosage_terms(cvd)
crp_add = add_dosage_terms(crp)
if not cvd_add.empty:
    save(cvd_add, "04_cvd_additive_genotype_model_dataset.csv")
if not crp_add.empty:
    save(crp_add, "05_crp_additive_genotype_model_dataset.csv")

join_rows = []
concordance_rows = []
qc_rows = []
cell_rows = []
for dataset_name, d in [("cvd", cvd_add), ("crp", crp_add)]:
    if d.empty:
        join_rows.append({"run_id": RUN_ID, "dataset": dataset_name, "rows": 0, "status": "missing_dataset"})
        continue
    join_row = {"run_id": RUN_ID, "dataset": dataset_name, "rows": len(d), "status": "merged" if not genotype_wide.empty else "no_genotype"}
    for rsid in TIER1_RSIDS:
        dosage = f"{rsid}_alt_dosage"
        if dosage not in d.columns:
            join_row[f"{rsid}_dosage_nonmissing_n"] = 0
            qc_rows.append({"run_id": RUN_ID, "dataset": dataset_name, "rsid": rsid, "status": "missing_dosage_col"})
            continue
        x = pd.to_numeric(d[dosage], errors="coerce")
        called = x.dropna()
        join_row[f"{rsid}_dosage_nonmissing_n"] = int(x.notna().sum())
        join_row[f"{rsid}_dosage_nonmissing_pct"] = pct(int(x.notna().sum()), len(d))
        n_called = int(len(called))
        alt_af = float(called.sum() / (2 * n_called)) if n_called else np.nan
        qc_rows.append({
            "run_id": RUN_ID, "dataset": dataset_name, "rsid": rsid,
            "status": "pass" if n_called else "no_called_genotype",
            "n_total": len(d), "n_called": n_called, "n_missing": int(x.isna().sum()),
            "missing_pct": pct(int(x.isna().sum()), len(d)),
            "genotype_0_ref_ref": int((called == 0).sum()),
            "genotype_1_ref_alt": int((called == 1).sum()),
            "genotype_2_alt_alt": int((called == 2).sum()),
            "alt_af": alt_af, "maf": min(alt_af, 1 - alt_af) if pd.notna(alt_af) else np.nan,
        })
        if "vitd_lt20" in d.columns:
            tmp = d.assign(alt_dosage=pd.to_numeric(d[dosage], errors="coerce"))
            for (vitd_lt20, alt_dosage), sub in tmp.groupby(["vitd_lt20", "alt_dosage"], dropna=False):
                row = {"run_id": RUN_ID, "dataset": dataset_name, "rsid": rsid, "vitd_lt20": vitd_lt20, "alt_dosage": alt_dosage, "n": len(sub)}
                if dataset_name == "cvd" and "event_mi_or_stroke" in sub.columns:
                    row["events"] = int(pd.to_numeric(sub["event_mi_or_stroke"], errors="coerce").fillna(0).sum())
                cell_rows.append(row)
        old = f"{rsid}_carrier"
        new = f"{rsid}_alt_carrier_from_dosage"
        if old in d.columns and new in d.columns:
            comp = d[[old, new]].dropna().copy()
            if len(comp):
                mismatch = int((pd.to_numeric(comp[old], errors="coerce").astype(int) != pd.to_numeric(comp[new], errors="coerce").astype(int)).sum())
                concordance_rows.append({"run_id": RUN_ID, "dataset": dataset_name, "rsid": rsid, "compared_n": len(comp), "mismatch_n": mismatch, "mismatch_pct": pct(mismatch, len(comp))})
    join_rows.append(join_row)

save(pd.DataFrame(join_rows), "06_additive_genotype_join_summary.csv")
save(pd.DataFrame(concordance_rows), "07_dosage_vs_existing_carrier_concordance.csv")
save(pd.DataFrame(qc_rows), "08_additive_genotype_analytic_qc.csv")
save(pd.DataFrame(cell_rows), "09_additive_vitd_dosage_cell_counts.csv")
